Imports

In [1]:
%load_ext autoreload
%autoreload 2

Data

In [ ]:
import pandas as pd
import numpy as np

# Buat comparison
from sklearn.naive_bayes import GaussianNB   
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression as LG

# buat data / preprocess
from sklearn.preprocessing import StandardScaler
X_train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

X_train

,id,num_adults,num_children,weekend_nights,weekday_nights,meal_package,needs_parking,room_category,days_before_arrival,arrival_year,arrival_month,arrival_day,booking_channel,is_returning_guest,prior_cancellations,prior_completed_bookings,nightly_rate_idr,special_requests,is_canceled
0,57152f82,2.0,NaN,1,2,Breakfast,0.0,Type B,88.0,2024,4,3,Offline Agent,0,0,0,1330000.0,0.0,1
1,af2ec71b,2.0,0.0,2,1,Half Board,0.0,Type D,120.0,2023,8,1,Offline Agent,0,0,0,NaN,0.0,0
2,5d5469ab,3.0,0.0,1,2,Breakfast,0.0,Type G,209.0,2024,8,28,Online Travel Agent,0,0,0,2099000.0,0.0,1
3,d2df0369,2.0,1.0,2,1,Breakfast,0.0,Type G,93.0,2024,8,13,Online Travel Agent,0,0,0,1693000.0,2.0,0
4,91d253db,2.0,0.0,0,1,Room Only,0.0,Type B,14.0,2024,9,1,Online Travel Agent,0,0,0,2010000.0,2.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27368,60213561,2.0,0.0,2,0,Room Only,0.0,Type B,2.0,2024,8,27,Online Travel Agent,0,0,0,1429000.0,0.0,0
27369,15354b3d,2.0,0.0,2,2,Room Only,0.0,Type B,NaN,2024,12,22,Online Travel Agent,0,0,0,1205000.0,1.0,1
27370,530d530c,2.0,0.0,0,2,Breakfast,0.0,Type C,115.0,2024,4,28,Online Travel Agent,0,0,0,1795000.0,0.0,1
27371,1d5d9c6a,2.0,NaN,0,1,Breakfast,0.0,Type B,58.0,2024,4,5,Offline Agent,0,0,0,1305000.0,0.0,0


Library

In [3]:
import kagglemethis.DTL as DTL
import kagglemethis.LogisticRegression as LR
import kagglemethis.NaiveBayes as NB

from itertools import product

Pre Processing - lowkey buns masihan

In [ ]:
X = X_train.drop(columns=["id", "is_canceled"])
y = X_train["is_canceled"]

cat_cols = X.select_dtypes(exclude=np.number).columns
num_cols = X.select_dtypes(include=np.number).columns

medians = X[num_cols].median()

# train pre process
X[cat_cols] = X[cat_cols].fillna("Missing")
X[num_cols] = X[num_cols].fillna(medians)
X = pd.get_dummies(X, columns=cat_cols, dtype=int)

# test pre process
X_test = test.drop(columns=["id"])
X_test[cat_cols] = X_test[cat_cols].fillna("Missing")
X_test[num_cols] = X_test[num_cols].fillna(medians)
X_test = pd.get_dummies(X_test, columns=cat_cols, dtype=int)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# scaling
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X.shape)
print(y.shape)
print(X_test.shape)

(27373, 32)
(27373,)
(6844, 32)


Model Declaration & Initial Accuracy 

In [5]:
dtl = DTL(max_depth=5)
dtl.fit(X, y)
dtl_scikit = DecisionTreeClassifier(max_depth=5)
dtl_scikit.fit(X,y)

print("Training accuracy DTL:", dtl.score(X, y))
print("Training accuracy DTL-scikit", dtl_scikit.score(X,y))

Training accuracy DTL: 0.8232930259745004
Training accuracy DTL-scikit 0.8251561757936653


In [6]:
logistic = LR()
logistic.fit(X, y)
logistic_scikit = LG()
logistic_scikit.fit(X,y)

print("Training accuracy LR:", logistic.score(X, y))
print("Training accuracy LR-scikit:", logistic_scikit.score(X, y))

Training accuracy LR: 0.34252730793117303
Training accuracy LR-scikit: 0.7549410002557264


c:\Users\Raysha\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
naive_bayes = NB()
naive_bayes.fit(X,y,categorical_features=[4,5,6,11,12])
naive_scikit = GaussianNB()
naive_scikit.fit(X, y)

print("Training accuracy NB:", naive_bayes.score(X, y))
print("Training accuracy NB-scikit:", naive_scikit.score(X, y))

Training accuracy NB: 0.39458590581960323
Training accuracy NB-scikit: 0.7607861761589888


Tuning

In [ ]:
def tuning(model, grid, X, y, cv=5, patience_threshold=0.01):
    param = list(grid.keys())
    val = list(grid.values())

    best_score = 0
    best_params = {}

    # shuffle indices once, reuse 
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    folds = np.array_split(indices, cv)

    for p in product(*val):
        params = dict(zip(param, p))

        scores = []
        prune = False

        for i in range(cv):
            val_idx   = folds[i]
            train_idx = np.concatenate([folds[j] for j in range(cv) if j != i])

            m = model.__class__(**params)
            m.fit(X[train_idx], y[train_idx])
            scores.append(m.score(X[val_idx], y[val_idx]))

            if i >= 1:
                projected = np.mean(scores)
                if projected < best_score - patience_threshold:
                    prune = True
                    break

        if prune:
            continue

        score = np.mean(scores)
        if score > best_score:
            best_score  = score
            best_params = params
            print(f"  New best: {best_score:.4f} — {best_params}")

    print(f"\nBest score : {best_score:.4f}")
    print(f"Best params: {best_params}")

    best_model = model.__class__(**best_params)
    best_model.fit(X, y)
    return best_model

In [9]:
dtl_params = {
    'max_depth':         [3, 5, 7, 10, None],
    'criterion':         ['entropy', 'gini'],
    'min_samples_split': [2, 10, 20, 50],
    'min_samples_leaf':  [1, 5, 10, 20],
}

tuned_dtl = tuning(dtl, dtl_params, X.values, y.values)

  New best: 0.7688 — {'max_depth': 3, 'criterion': 'entropy', 'min_samples_split': 2, 'min_samples_leaf': 1}
  New best: 0.7696 — {'max_depth': 3, 'criterion': 'gini', 'min_samples_split': 2, 'min_samples_leaf': 1}
  New best: 0.8164 — {'max_depth': 5, 'criterion': 'entropy', 'min_samples_split': 2, 'min_samples_leaf': 1}
  New best: 0.8164 — {'max_depth': 5, 'criterion': 'entropy', 'min_samples_split': 2, 'min_samples_leaf': 5}
  New best: 0.8164 — {'max_depth': 5, 'criterion': 'entropy', 'min_samples_split': 50, 'min_samples_leaf': 1}
  New best: 0.8195 — {'max_depth': 5, 'criterion': 'gini', 'min_samples_split': 2, 'min_samples_leaf': 1}
  New best: 0.8195 — {'max_depth': 5, 'criterion': 'gini', 'min_samples_split': 2, 'min_samples_leaf': 5}
  New best: 0.8195 — {'max_depth': 5, 'criterion': 'gini', 'min_samples_split': 2, 'min_samples_leaf': 10}
  New best: 0.8195 — {'max_depth': 5, 'criterion': 'gini', 'min_samples_split': 50, 'min_samples_leaf': 1}
  New best: 0.8296 — {'max_dept

In [12]:
lr_params = {
    'rate': [0.001, 0.01, 0.1, 1.0],
    'epochs':      [100, 500, 1000],
    'penalty':       ['l2', None],
}

tuned_lr = tuning(logistic, lr_params, X.values, y.values)


  New best: 0.4814 — {'rate': 0.001, 'epochs': 100, 'penalty': 'l2'}
  New best: 0.6723 — {'rate': 0.001, 'epochs': 1000, 'penalty': 'l2'}

Best score : 0.6723
Best params: {'rate': 0.001, 'epochs': 1000, 'penalty': 'l2'}


In [15]:
nb_params = {
    'var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3],
    'alpha': [0.1, 0.5, 1.0, 2.0],
}

tuned_nb = tuning(naive_bayes, nb_params, X.values, y.values)

  New best: 0.7607 — {'var_smoothing': 1e-09, 'alpha': 0.1}

Best score : 0.7607
Best params: {'var_smoothing': 1e-09, 'alpha': 0.1}


Submit

In [17]:
model = tuned_nb

def submit(model, X_test, test):
    predictions = model.predict(X_test)

    result = pd.DataFrame({
        "id": test["id"],
        "is_canceled": predictions
    })

    result.to_csv("data/submission.csv", index=False)
    return result.head()

submit(model, X_test, test)

,id,is_canceled
0,5cb6f473,1
1,c06b4ab3,1
2,22bf3fc3,0
3,25408a48,0
4,621586b6,0
